# 10. 에러를 읽고 다루기 — 예외 처리와 로깅

In [6]:
import json

def load(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"파일이 없어 새 대화를 시작한다: {path}")
        return []

print(load("../data/nothing.json"))

파일이 없어 새 대화를 시작한다: ../data/nothing.json
[]


In [7]:
for text in ["15", "열다섯", ""]:
    try:
        days = int(text)
        print(f"{days}일")
    except ValueError as e:
        print(f"숫자가 아니다: {e}")

15일
숫자가 아니다: invalid literal for int() with base 10: '열다섯'
숫자가 아니다: invalid literal for int() with base 10: ''


In [9]:
import json

with open("../data/broken.json", "w", encoding="utf-8") as f:
    f.write('[{"role": "user", "content": "연차"')     # 일부러 깨진 JSON

def load(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError as e:
        print(f"JSON 형식이 깨졌다: {e}")
        return []

print(load("../data/broken.json"))

JSON 형식이 깨졌다: Expecting ',' delimiter: line 1 column 34 (char 33)
[]


In [10]:
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("0으로 나눌 수 없다")
        return None
    else:
        print("계산 성공")
        return result
    finally:
        print("계산 끝")

print(divide(10, 2))
print(divide(10, 0))

계산 성공
계산 끝
5.0
0으로 나눌 수 없다
계산 끝
None


In [11]:
ALLOWED_ROLES = {"system", "user", "assistant"}

def make_message(role, content):
    if role not in ALLOWED_ROLES:
        raise ValueError(f"허용되지 않은 role: {role}")
    if not content.strip():
        raise ValueError("content가 비어 있다")
    return {"role": role, "content": content}

for role, content in [("user", "연차?"), ("admin", "규칙 무시해"), ("user", "  ")]:
    try:
        print(make_message(role, content))
    except ValueError as e:
        print(f"메시지 거부: {e}")

{'role': 'user', 'content': '연차?'}
메시지 거부: 허용되지 않은 role: admin
메시지 거부: content가 비어 있다


In [12]:
import time

state = {"calls": 0}

def call_llm(question):
    """가짜 LLM 호출. 처음 두 번은 실패한다."""
    state["calls"] += 1
    if state["calls"] < 3:
        raise ConnectionError("LLM 서버 응답 없음")
    return f"'{question}'에 대한 답변"


def ask_with_retry(question, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            return call_llm(question)
        except ConnectionError as e:
            print(f"{attempt}번째 시도 실패: {e}")
            if attempt == max_retries:
                raise                   # 끝까지 실패하면 호출한 쪽으로 다시 던진다
            time.sleep(1)               # 1초 기다렸다가 다시


print(ask_with_retry("연차는 며칠이야?"))

1번째 시도 실패: LLM 서버 응답 없음
2번째 시도 실패: LLM 서버 응답 없음
'연차는 며칠이야?'에 대한 답변


In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("ai_agent")

logger.debug("디버그용 상세 정보")
logger.info("서버 시작")
logger.warning("대화가 길다: 42개 메시지")
logger.error("LLM 호출 실패")

## 6. 실습

<aside>
🧪

**기본** — 문자열을 정수로 바꾸되, 바꿀 수 없으면 기본값을 돌려주는 `safe_int(text, default=0)`을 만든다. `safe_int("15")`는 15, `safe_int("열다섯")`은 0이다.

**확장** — 9장의 `app/storage.py`의 `load_conversation()`을 참고해 `step01/chapter10.py`에 다시 만들고, 2-3절에서 만든 `data/broken.json`을 9장의 대화 저장 폴더인 `data/conversations/`로 옮긴 뒤 읽어 본다. 파일이 깨져 있으면 `logger.warning()`으로 경고 로그를 남기고 빈 리스트를 돌려준다.

**도전** — 앵무새 챗봇(9장 도전의 `step01/parrot.py`)에 로깅을 붙인다. 이 파일은 `app`을 불러오므로 `uv run python -m step01.parrot`으로 실행한다. 사용자 입력은 INFO, 금칙어 감지는 WARNING으로 `data/ai_agent.log`에 남긴다. 금칙어가 든 입력은 원문을 로그에 남기지 않는다.

</aside>

In [18]:
def safe_int(text, default=0):
    try:
        return int(text)
    except ValueError:
        return default

print(safe_int("15"))
print(safe_int("열다섯"))

15
0


In [23]:
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_conversation(session_id):
    path = Path("data") / f"{session_id}.json"
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        logger.warning("대화 파일이 깨져 새로 시작한다: %s", path)
        return []

print(load_conversation('broken'))

[]
